# 用 GPT 做影評情緒分類：用「提示詞」呼叫雲端模型的 API

---

## 這份程式要做什麼

這是你這一系列「影評情緒分類」的**第五種做法**：不在自己的電腦上跑模型，而是**透過網路呼叫雲端上的 GPT**，用一段文字指示（提示詞）叫它做分類。

```
提示詞模板 + 一篇影評
        │  把 [DOCUMENT] 換成影評
        ▼
一份完整的指示  ──透過網路（API）──▶  OpenAI 的 GPT 模型
                                           │
                                           ▼
                              回覆：  "1"（正面）或 "0"（負面）
```

## 五種做法的位置

| 做法 | 模型在哪裡跑 | 需要標註資料 | 訓練 |
|---|---|---|---|
| ① 微調 BERT | 你的電腦 | 多 | 大 |
| ② 凍結 Embedding + 邏輯回歸 | 你的電腦 | 多 | 小 |
| ③ 現成的任務專用模型 | 你的電腦 | 不需要 | 無 |
| ④ 零樣本（標籤描述 + 向量相似度） | 你的電腦 | 不需要 | 無 |
| **⑤ 生成式模型 + 提示詞（這份）** | **雲端（OpenAI）** | **不需要** | **無** |

前四種的模型都**下載到你的電腦**執行；這一種的模型**根本不在你的電腦上**，你只是送一段文字出去、收一段文字回來。

## 這個做法有什麼不同

| 面向 | 本機模型（前四種） | **雲端 API（這份）** |
|---|---|---|
| 硬體需求 | 需要 GPU 或耐心等 | **不需要**，任何電腦都能跑 |
| 費用 | 電費（免費下載模型） | **按用量計費**（每個 token 都要錢） |
| 隱私 | 資料不離開你的電腦 | **資料會傳到第三方伺服器** |
| 網路 | 載入後可離線 | **每次都要連網** |
| 模型能力 | 受限於你能跑的大小 | 可使用很大的模型 |
| 穩定性 | 版本固定 | 模型可能被更新或**停用** |

## 先搞懂幾個關鍵詞

| 名稱 | 意思 |
|---|---|
| **API** | 一個讓程式透過網路呼叫別人服務的介面 |
| **API 金鑰（API key）** | 證明「你是誰、要記到誰的帳上」的一串密碼 |
| **Client** | 程式裡負責替你連線、送出請求的物件 |
| **提示詞（prompt）** | 你寫給模型的指示 |
| **溫度（temperature）** | 控制回答的隨機程度，0 代表最確定 |
| **token** | 計費與長度的單位，約等於一個字或半個字 |

## 步驟 0：載入套件

| 匯入 | 用途 |
|---|---|
| `openai` | OpenAI 的官方套件。這份程式**沒有直接用到**這個匯入（下一行才是真正用到的） |
| `load_dotenv`（來自 `python-dotenv`） | 從 `.env` 檔案讀取設定，主要是 API 金鑰 |
| `OpenAI` | 建立 client 的類別 |

需要先安裝：`pip install openai python-dotenv`。

In [1]:
import openai
from dotenv import load_dotenv
from openai import OpenAI

## 步驟 1：載入金鑰並建立 client

```python
load_dotenv()
client = OpenAI()
```

### `load_dotenv()` 在做什麼

在你的專案資料夾裡放一個叫 `.env` 的文字檔，內容像這樣：

```
OPENAI_API_KEY=你的金鑰
```

`load_dotenv()` 會讀取這個檔案，把裡面的設定放進**環境變數**。

### `OpenAI()` 為什麼不用給金鑰

建立 client 時，OpenAI 的套件會**自動去環境變數 `OPENAI_API_KEY` 找金鑰**。所以只要 `load_dotenv()` 先把金鑰放進去，`OpenAI()` 就能直接用。這也是金鑰**不會出現在程式碼裡**的原因。

| 寫法 | 安全嗎 |
|---|---|
| `OpenAI(api_key="sk-abc123...")` 直接寫在程式裡 | ❌ **危險**：程式一分享、一上傳 Git，金鑰就外洩了 |
| `.env` + `load_dotenv()` | ✅ 金鑰只在你的電腦上 |


In [2]:
load_dotenv()
client = OpenAI()

## 步驟 2：定義呼叫函式

```python
def chatgpt_generation(prompt, document, model="gpt-3.5-turbo-0125"):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are a sentiment classifier. ..."},
            {"role": "user", "content": prompt.replace("[DOCUMENT]", document)},
        ],
        temperature=0,
    )
    return response.choices[0].message.content
```

### 參數說明

| 參數 | 說明 |
|---|---|
| `prompt` | **提示詞模板**，裡面有一個佔位符 `[DOCUMENT]` |
| `document` | **要分類的影評** |
| `model` | 要用哪個模型。有**預設值**，呼叫時可以換成別的 |

### `messages`：對話是一串「角色 + 內容」

聊天模型接收的是一個**訊息清單**，每則訊息有 `role`（誰說的）和 `content`（說了什麼）：

| role | 意思 | 這份程式裡 |
|---|---|---|
| `system` | **系統提示**：設定模型的整體角色、規則、行為 | 「你是情緒分類器，只回答 positive 或 negative」 |
| `user` | **使用者說的話** | 填好影評的提示詞 |
| `assistant` | 模型的回答（多輪對話時，把上一輪的回答放回來） | 這裡沒有 |

你的註解寫得很準：`system` 是「設定模型的整體角色／規則／行為」。

### `prompt.replace("[DOCUMENT]", document)`

這是最簡單的「**模板填空**」：把提示詞裡的 `[DOCUMENT]` 換成真正的影評。

好處：**模板只寫一次**，之後換成任何影評都能用，不用重寫提示詞。

### `temperature=0`

| temperature | 效果 |
|---|---|
| **0** | 每一步都選機率最高的字，回答最穩定、最可重現 |
| 較高（例如 0.8） | 更隨機、更有創意 |

做**分類**時要的是穩定、一致的答案，所以用 0。需要注意：溫度 0 **大幅減少**隨機性，但我印象中**不保證**每次輸出完全相同（我沒有針對你的模型確認）。

### `response.choices[0].message.content`

你的註解說得很好：「response 通常不是字串，而是很多欄位的回傳物件，可以透過 print() 查看」。

```
ChatCompletion(id='chatcmpl-demo', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='1', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1700000000, model='gpt-3.5-turbo-0125', object='chat.completion', metadata=None, moderation=None, service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=1, prompt_tokens=120, total_tokens=121, completion_tokens_details=None, prompt_tokens_details=None))
```

**逐層看懂 `response.choices[0].message.content`：**

| 路徑 | 意思 |
|---|---|
| `response` | 整個回傳物件 |
| `.choices` | **候選回答的清單**（預設只有 1 個，所以是長度 1 的 list） |
| `[0]` | 取第一個候選回答 |
| `.message` | 這個回答的訊息（含 `role` 與 `content`） |
| `.content` | **模型實際說的話，是一個字串** |

其他有用的欄位：

| 欄位 | 意思 |
|---|---|
| `finish_reason` | 為什麼停下來：`stop` 是自然結束，`length` 是被字數上限截斷 |
| `usage` | **用了多少 token**：`prompt_tokens`（送出的）、`completion_tokens`（回來的）。**費用就是依這個算的** |
| `model` | 實際回應的模型版本 |

In [3]:
def chatgpt_generation(prompt, document, model="gpt-3.5-turbo-0125"):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are a sentiment classifier. Respond only with 'positive' or 'negative'."},   #設定模型的整體角色/規則/行為
            {"role": "user", "content": prompt.replace("[DOCUMENT]", document)},
        ],
        temperature=0,
    )
    return response.choices[0].message.content    #response通常不是字串 而是很多欄位的回傳物件 可以透過print()查看

## 步驟 3：寫提示詞模板

```python
prompt = """Predict wheter the following document is a positive or negative movie review.:

[DOCUMENT]

If it is a positive review, please respond with "1". 
If it is a negative review, please respond with "0". 
Please do not give any other answer.
"""
```

### 這個提示詞的結構

| 部分 | 作用 |
|---|---|
| `Predict whether ... movie review.:` | **任務說明**：告訴模型要做什麼 |
| `[DOCUMENT]` | **佔位符**：等一下會被換成影評 |
| `please respond with "1" / "0"` | **輸出格式**：規定答案只能是 1 或 0 |
| `Please do not give any other answer.` | **限制**：禁止多餘的解釋 |

### 為什麼要要求「只回答 1 或 0」

模型很愛多說話（例如「這是一篇正面的影評，因為……」）。分類時我們只要一個標籤，所以**嚴格限制輸出格式**，程式才容易處理：

```python
int(chatgpt_response)       # "1" → 1，"0" → 0
```

### ⚠️ 這份程式有兩條互相衝突的指示

| 位置 | 指示 |
|---|---|
| **system 訊息**（步驟 2） | 「Respond only with **'positive' or 'negative'**」 |
| **user 訊息**（這個提示詞） | 「respond with **"1"** or **"0"**」 |

系統提示要它回答**文字**，使用者提示要它回答**數字**。兩個指示不一致時，模型的行為不一定（我沒有實測它會選哪個）。常見的處理是兩邊的輸出格式保持一致。要得到 `"1"` / `"0"`，就把系統提示改成同樣的要求（見實驗區）。

### 小細節

- `wheter` 是 `whether` 的筆誤，對模型通常影響不大，但最好改正
- 提示詞裡有兩行結尾多了空格，不影響結果
- 三重引號字串會**保留所有換行**，所以提示詞的排版就是你看到的樣子

In [4]:
prompt = """Predict wheter the following document is a positive or negative movie review.:

[DOCUMENT]

If it is a positive review, please respond only with "positive". 
If it is a negative review, please respond only with "negative". 
Please do not give any other answer.
"""

## 步驟 4：準備要分類的影評

```python
document = "This movie is a great movie. The story is very interesting and the acting is superb. ..."
```

一段明顯是**正面**的影評（「很棒的電影」「故事很有趣」「演技一流」「強烈推薦」）。拿一個答案很明確的例子測試，容易判斷模型有沒有答對：**預期回答 `"1"`**。

In [5]:
document = "This movie is a great movie. The story is very interesting and the acting is superb. I would highly recommend it to anyone who enjoys a good film."

## 步驟 5：呼叫並印出結果

```python
chatgpt_response = chatgpt_generation(prompt, document)
print(chatgpt_response)
```

### 這一步會發生什麼

```
① 把影評填進提示詞
② 透過網路送到 OpenAI
③ OpenAI 的模型產生回答
④ 回傳物件，取出 .content
⑤ print 出來
```

**這一步要連網、會花錢、需要金鑰。** 每呼叫一次，就會依用到的 token 數計費。

### 一個重要的細節：回答是**字串**

`chatgpt_response` 是字串 `"1"`，**不是整數 `1`**。要拿去和資料集的標籤（整數 0、1）比較，必須轉換：

```python
int(chatgpt_response)
```

| | 型別 | 能直接和標籤比較嗎 |
|---|---|---|
| `"1"` | 字串 | ❌ `"1" == 1` 是 `False` |
| `1` | 整數 | ✅ |

### 這份程式只測了一則影評

只用一篇影評當作示範，**無法評估模型的整體表現**。要得到像前四種做法那樣的分類報告，必須對整個測試集（1066 筆）逐筆呼叫，見實驗區。

In [6]:
chatgpt_response = chatgpt_generation(prompt, document)

print(chatgpt_response)

positive


## 如果要像前四種做法一樣完整評估

這份只做了「**單筆**」，要得到分類報告，需要對整個測試集跑迴圈：

```
對測試集每一筆影評：
   呼叫 API → 取得回答字串 → 轉成整數 → 存進 y_pred
最後 evaluate_performance(真實標籤, y_pred)
```

需要特別考量的事：

| 考量 | 說明 |
|---|---|
| **費用** | 1066 次呼叫，每次都按 token 計費。先用少量資料（例如 20 筆）測試 |
| **時間** | 每次呼叫要等網路來回，1066 次可能要好幾分鐘 |
| **速率限制** | API 有每分鐘請求數上限，太快會被拒絕（要重試或放慢） |
| **輸出驗證** | 模型可能回 `"1."`、`"The answer is 1"` 之類的，轉整數會失敗，需要檢查 |
| **錯誤處理** | 網路中斷、逾時、額度用完，都要能應付 |

這些在實驗區有範例寫法。

---

# 重點整理

## 一、整份程式一句話

> 用 `.env` 載入 API 金鑰、建立 OpenAI client，把一則影評填進提示詞模板，連同「你是情緒分類器」的系統提示，透過網路送給雲端的 GPT，並從回傳物件取出模型的回答字串（預期是代表正面的 `"1"`）。

## 二、整條資料流

| 步驟 | 程式 | 結果 |
|---|---|---|
| 載入金鑰 | `load_dotenv()` | 金鑰進入環境變數 |
| 建立 client | `OpenAI()` | 自動讀取環境變數中的金鑰 |
| 填模板 | `prompt.replace("[DOCUMENT]", document)` | 完整的使用者訊息 |
| 組訊息 | `messages=[system, user]` | 一份「角色 + 內容」的清單 |
| 呼叫 API | `client.chat.completions.create(...)` | 回傳物件 |
| 取出答案 | `response.choices[0].message.content` | 字串 `"1"` 或 `"0"` |

## 三、必須搞懂的 10 個概念

| # | 概念 | 一句話解釋 |
|---|---|---|
| 1 | API | 讓程式透過網路呼叫別人服務的介面 |
| 2 | API 金鑰 | 證明身分與計費對象的密碼，絕不能外洩 |
| 3 | `.env` + `load_dotenv` | 把金鑰放在程式之外，避免寫進程式碼 |
| 4 | 角色（role） | `system` 設定規則、`user` 說話、`assistant` 回答 |
| 5 | 提示詞模板 | 用佔位符寫一次，重複填入不同內容 |
| 6 | `temperature=0` | 最確定、最穩定的輸出，適合分類 |
| 7 | 回傳物件 | 不是字串，要用 `.choices[0].message.content` 取出 |
| 8 | token 與計費 | `usage` 顯示用量，費用依此計算 |
| 9 | 輸出格式限制 | 要求「只回 0 或 1」，程式才好處理 |
| 10 | 回答是字串 | 要轉成整數才能和標籤比較 |

## 四、最容易搞混的地方

1. **模型在我的電腦上跑嗎？**
   不是。程式只送文字出去、收文字回來，模型在雲端。

2. **金鑰為什麼不寫在程式裡？**
   程式會被分享、上傳，金鑰一旦外洩就會被盜用，費用算在你的帳上。

3. **`system` 和 `user` 有什麼不同？**
   `system` 設定模型整體的角色與規則；`user` 是這一次要它處理的內容。

4. **`response` 為什麼不能直接 print 出答案？**
   它是包含很多欄位的物件，答案藏在 `choices[0].message.content`。

5. **`choices` 為什麼是 list？**
   API 可以一次要求多個候選回答，預設只有一個，所以是長度為 1 的 list。

6. **回答的 `"1"` 能直接當預測嗎？**
   不能，它是字串，要先 `int()`。

7. **`temperature=0` 就保證結果一樣嗎？**
   大幅降低隨機性，但不保證每次完全相同。

8. **為什麼 system 與 user 的指示不一致是問題？**
   一個說回答 positive／negative，一個說回答 1／0，模型不一定會選你想要的。

## 五、常見錯誤排查表

| 症狀 | 可能原因 | 處理 |
|---|---|---|
| `OpenAIError: The api_key client option must be set` | 找不到金鑰 | 確認 `.env` 檔案存在、內容是 `OPENAI_API_KEY=...`，且在專案資料夾下執行 |
| `AuthenticationError`（401） | 金鑰錯誤或已撤銷 | 重新產生金鑰 |
| `NotFoundError`（模型不存在） | 模型名稱已停用或打錯 | 改用目前可用的模型名稱（見開頭提醒） |
| `RateLimitError`（429） | 呼叫太快，或額度、用量上限用完 | 放慢速度、加重試、檢查帳戶額度 |
| 連線逾時、`APIConnectionError` | 網路問題 | 檢查網路，加重試機制 |
| 回答不是純 `0` 或 `1` | 模型多話，或 system 與 user 指示衝突 | 統一兩邊的格式要求；加上輸出驗證 |
| `int(...)` 轉換失敗 | 回答含其他文字 | 先 `.strip()`，驗證後再轉換 |
| 費用比預期高 | 提示詞太長、呼叫次數多 | 看 `usage`；先用少量資料測試 |

## 六、觀察到的幾個地方（程式碼未修改，僅供參考）

1. **system 與 user 的輸出格式衝突**
   system 說回答 `positive` / `negative`，user 說回答 `1` / `0`。

2. **模型 `gpt-3.5-turbo-0125` 可能即將停用**
   第三方資料顯示預計 2026 年 10 月 23 日終止，建議的替代是 `gpt-4.1-mini`。請以官方公告為準。

3. **`import openai` 沒有用到**
   只有 `from openai import OpenAI` 被用到。

4. **只測了一則影評**
   無法評估整體表現，也無法和前四種做法比較。

5. **沒有錯誤處理**
   網路、速率限制、額度問題都沒有處理。

6. **回答沒有驗證**
   直接拿來用，沒有檢查是不是 `"0"` 或 `"1"`。

7. **`temperature=0` 不保證完全一致**
   同樣的輸入，不同時間可能有細微差異。

8. **資料會傳到第三方**
   影評如果是敏感內容，要考慮隱私。

9. **筆誤 `wheter`**
   應為 `whether`，通常不影響結果。

10. **模型名稱寫死在函式預設值**
    換模型時要改函式，或每次呼叫時傳入 `model=`。

11. **Chat Completions 之外還有較新的介面**
    OpenAI 另有較新的 API，Chat Completions 的長期狀態請查官方文件，我沒有確認。

## 七、可以自己做的實驗（需要時再自己加，不影響原程式）

```python
# 1. 看完整的回傳物件（你的註解也建議這樣做）
resp = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[{"role": "user", "content": "Say hi"}],
)
print(resp)
print(resp.usage)                 # 用了多少 token

# 2. 讓 system 與 user 的輸出格式一致
system_msg = "You are a sentiment classifier. Respond only with '1' for positive or '0' for negative."

# 3. 完整評估整個測試集（先用少量資料測試！）
from datasets import load_dataset
from tqdm import tqdm
data = load_dataset("cornell-movie-review-data/rotten_tomatoes")
y_pred = []
for text in tqdm(data["test"]["text"][:20]):          # 先只跑 20 筆
    answer = chatgpt_generation(prompt, text).strip()
    y_pred.append(int(answer) if answer in ("0", "1") else 0)   # 輸出驗證；不合法的先當作 0
# evaluate_performance(data["test"]["label"][:20], y_pred)

# 4. 加上重試，應付速率限制與網路錯誤
import time
def safe_call(prompt, document, retries=3):
    for attempt in range(retries):
        try:
            return chatgpt_generation(prompt, document)
        except Exception as e:
            print("錯誤：", type(e).__name__, "，稍後重試")
            time.sleep(2 ** attempt)
    return None

# 5. 限制回答長度，省錢又快：只需要 1 個字元
# 在 create(...) 加上 max_tokens=1

# 6. 少樣本提示：給模型幾個範例，通常更準
few_shot_prompt = """Predict whether the following document is a positive or negative movie review.

Review: A dull and lifeless film.
Answer: 0

Review: An unforgettable and moving story.
Answer: 1

Review: [DOCUMENT]
Answer:"""

# 7. 換成較新的模型，比較結果
# chatgpt_generation(prompt, document, model="gpt-4.1-mini")

# 8. 從環境變數檢查金鑰是否設定（不要印出金鑰本身！）
import os
print("已設定金鑰:", "OPENAI_API_KEY" in os.environ)
```

## 八、五種影評情緒分類做法的對照

| | ① 微調 BERT | ② 凍結 Embedding + 邏輯回歸 | ③ 現成任務模型 | ④ 零樣本向量 | **⑤ 生成式 API（這份）** |
|---|---|---|---|---|---|
| 模型在哪裡 | 本機 | 本機 | 本機 | 本機 | **雲端** |
| 標註資料 | 多 | 多 | 不需要 | 不需要 | **不需要** |
| 訓練 | 大 | 小 | 無 | 無 | **無** |
| 費用 | 硬體電費 | 低 | 低 | 低 | **按次計費** |
| 隱私 | 資料留在本機 | 本機 | 本機 | 本機 | **資料傳到第三方** |
| 可調校的地方 | 超參數 | 分類器 | 幾乎沒有 | 標籤描述 | **提示詞** |
| 輸出的穩定性 | 高 | 高 | 高 | 高 | **需要驗證格式** |
| 靈活度 | 低（只能做訓練過的任務） | 低 | 低 | 中 | **高（改提示詞就能換任務）** |

## 九、怎麼選

| 你的情況 | 建議 |
|---|---|
| 資料敏感、不能外傳 | 本機模型（前四種） |
| 沒有 GPU、想快速試試 | **這份：雲端 API** |
| 任務經常改變，想靠改提示詞應付 | **生成式 API** |
| 大量資料、要控制成本 | 本機模型（每筆幾乎免費） |
| 追求在特定任務上的最佳效果 | 微調 |

## 十、一句話記住

> **用 API 呼叫雲端模型 = 寫好提示詞、送出去、取回答案。** 金鑰要保護、輸出要驗證、費用要留意；而「改提示詞就能換任務」是它最大的優勢。